# ETL : Limpieza y Estandarización (2018-2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.  
El objetivo es limpiar, normalizar y estandarizar los tres datasets del septenio aplicando los hallazgos documentados en los notebooks de análisis exploratorio (EDA), preparando los datos para las etapas posteriores de integración y modelado.


In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Funciones de Limpieza

In [2]:
# -- Función de normalización de nombres ----------------------------------------------
def normalizar_columnas(df):
    """Convierte nombres de columnas a minúscula sin tildes."""
    tildes = str.maketrans('áéíóúÁÉÍÓÚ', 'aeiouAEIOU')
    df.columns = [c.lower().translate(tildes) for c in df.columns]
    return df

# -- Prueba rápida ----------------------------------------------
test = pd.DataFrame(columns=['Núm_corre', 'Día_ocu', 'g_edad_80ymás'])
print(normalizar_columnas(test).columns.tolist())

['num_corre', 'dia_ocu', 'g_edad_80ymas']


## 2. ETL : Hechos de Tránsito

In [3]:
# -- Carga y limpieza hechos ----------------------------------------------
RAW_HECHOS = '../data/raw/ACCIDENTES DE TRANSITO - HECHOS'
YEARS = range(2018, 2025)

frames = []
for year in YEARS:
    path = os.path.join(RAW_HECHOS, f'hechos-de-transito-ano-{year}.xlsx')
    df_year = pd.read_excel(path)
    df_year = normalizar_columnas(df_year)
    df_year['año_carga'] = year
    frames.append(df_year)

df_hechos = pd.concat(frames, ignore_index=True)

# -- Eliminar columnas duplicadas (variantes del mismo campo) ----------------------------------------------
df_hechos = df_hechos.loc[:, ~df_hechos.columns.duplicated()]

# -- Eliminar zona_ciudad si existe (solo en 2021 y redundante) ----------------------------------------------
if 'zona_ciudad' in df_hechos.columns:
    df_hechos.drop(columns=['zona_ciudad'], inplace=True)

# -- Asegurar tipos de datos correctos (sin nulos reales) ----------------------------------------------
df_hechos['dia_ocu'] = df_hechos['dia_ocu'].astype(int)

# -- Verificar columnas finales ----------------------------------------------
print(f'Dimensiones: {df_hechos.shape}')
print(f'\nColumnas: {df_hechos.columns.tolist()}')

Dimensiones: (52488, 18)

Columnas: ['num_corre', 'año_ocu', 'dia_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu', 'zona_ocu', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', 'año_carga']


### ( Hechos normalizado ) : 18 columnas limpias, 52,488 registros

In [4]:
# -- Validar nulos tras limpieza ----------------------------------------------
nulos = df_hechos.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) == 0:
    print('✓ Sin nulos')
else:
    print(nulos)

# -- Validar no queden duplicados de columnas ----------------------------------------------
dupes = [c for c in df_hechos.columns if df_hechos.columns.tolist().count(c) > 1]
print(f'Columnas duplicadas: {dupes if dupes else "ninguna"}')

# -- Muestra ----------------------------------------------
print(f'\nPrimeras 3 filas:')
df_hechos.head(3)

✓ Sin nulos
Columnas duplicadas: ninguna

Primeras 3 filas:


,num_corre,año_ocu,dia_ocu,hora_ocu,g_hora,g_hora_5,mes_ocu,dia_sem_ocu,mupio_ocu,depto_ocu,zona_ocu,tipo_veh,marca_veh,color_veh,modelo_veh,g_modelo_veh,tipo_eve,año_carga
0,1,2018,1,16,3,2,1,1,115,1,99,4,32,5,9999,99,2,2018
1,2,2018,1,12,3,2,1,1,2207,22,99,1,69,2,9999,99,1,2018
2,3,2018,1,7,2,1,1,1,2102,21,99,1,999,6,9999,99,2,2018


### ( Hechos validado ) : datos correctos, sin nulos, sin duplicados

In [5]:
# -- Guardar hechos limpio ----------------------------------------------
OUT_PATH = '../data/clean'
os.makedirs(OUT_PATH, exist_ok=True)

df_hechos.to_csv(os.path.join(OUT_PATH, 'hechos_clean.csv'), index=False)
print(f'✓ hechos_clean.csv guardado — {df_hechos.shape[0]:,} registros × {df_hechos.shape[1]} columnas')

✓ hechos_clean.csv guardado — 52,488 registros × 18 columnas


## 3. ETL : Vehículos Involucrados

In [6]:
# -- Carga y limpieza vehículos involucrados ----------------------------------------------
RAW_VEH = '../data/raw/ACCIDENTES DE TRÁNSITO - VEHICULOS INVOLUCRADOS'

frames = []
for year in YEARS:
    path = os.path.join(RAW_VEH, f'vehiculos-involucrados-ano-{year}.xlsx')
    df_year = pd.read_excel(path)
    df_year = normalizar_columnas(df_year)
    df_year['año_carga'] = year
    frames.append(df_year)

df_veh = pd.concat(frames, ignore_index=True)

# -- Eliminar zona_ciudad ----------------------------------------------
if 'zona_ciudad' in df_veh.columns:
    df_veh.drop(columns=['zona_ciudad'], inplace=True)

# -- Eliminar columnas duplicadas que quedaron tras normalización ----------------------------------------------
df_veh = df_veh.loc[:, ~df_veh.columns.duplicated()]

# -- Verificar ----------------------------------------------
print(f'Dimensiones: {df_veh.shape}')
print(f'\nColumnas: {df_veh.columns.tolist()}')

Dimensiones: (80721, 25)

Columnas: ['num_corre', 'año_ocu', 'dia_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymas', 'g_edad_60ymas', 'edad_quinquenales', 'estado_con', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', 'año_carga']


### Vehículos involucrados normalizado : 25 columnas limpias, 80,721 registros

In [7]:
# -- Validar nulos tras limpieza ----------------------------------------------
nulos = df_veh.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) == 0:
    print('✓ Sin nulos')
else:
    print(nulos)

dupes = [c for c in df_veh.columns if df_veh.columns.tolist().count(c) > 1]
print(f'Columnas duplicadas: {dupes if dupes else "ninguna"}')

print(f'\nPrimeras 3 filas:')
df_veh.head(3)

✓ Sin nulos
Columnas duplicadas: ninguna

Primeras 3 filas:


,num_corre,año_ocu,dia_ocu,hora_ocu,g_hora,g_hora_5,mes_ocu,dia_sem_ocu,mupio_ocu,depto_ocu,zona_ocu,sexo_per,edad_per,g_edad_80ymas,g_edad_60ymas,edad_quinquenales,estado_con,mayor_menor,tipo_veh,marca_veh,color_veh,modelo_veh,g_modelo_veh,tipo_eve,año_carga
0,1,2018,1,16,3,2,1,1,115,1,99,1,28,4,4,6,9,1,4,32,5,9999,99,2,2018
1,2,2018,1,12,3,2,1,1,2207,22,99,1,19,2,2,4,1,1,1,69,2,9999,99,1,2018
2,3,2018,1,12,3,2,1,1,2207,22,99,1,60,11,11,13,1,1,4,6,1,9999,99,1,2018


### ( Vehículos involucrados validado ) : sin nulos, sin duplicados

In [8]:
# -- Guardar vehículos limpio ----------------------------------------------
df_veh.to_csv(os.path.join(OUT_PATH, 'vehiculos_involucrados_clean.csv'), index=False)
print(f'✓ vehiculos_involucrados_clean.csv guardado — {df_veh.shape[0]:,} registros × {df_veh.shape[1]} columnas')

✓ vehiculos_involucrados_clean.csv guardado — 80,721 registros × 25 columnas


## 4. ETL : Fallecidos y Lesionados

In [9]:
# -- Carga y limpieza fallecidos y lesionados ----------------------------------------------
RAW_FL = '../data/raw/ACCIDENTES DE TRANSITO - FALLECIDOS Y LESIONADOS'

frames = []
for year in YEARS:
    path = os.path.join(RAW_FL, f'fallecidos-y-lesionados-ano-{year}.xlsx')
    df_year = pd.read_excel(path)
    df_year = normalizar_columnas(df_year)
    df_year['año_carga'] = year
    frames.append(df_year)

df_fl = pd.concat(frames, ignore_index=True)

# -- Eliminar zona_ciudad ----------------------------------------------
if 'zona_ciudad' in df_fl.columns:
    df_fl.drop(columns=['zona_ciudad'], inplace=True)

# -- Eliminar columnas duplicadas ----------------------------------------------
df_fl = df_fl.loc[:, ~df_fl.columns.duplicated()]

# -- Eliminar 9 registros con fall_les == 9 (código no documentado) ----------------------------------------------
antes = len(df_fl)
df_fl = df_fl[df_fl['fall_les'] != 9].reset_index(drop=True)
print(f'Registros eliminados (fall_les=9): {antes - len(df_fl)}')

# -- Verificar ----------------------------------------------
print(f'Dimensiones: {df_fl.shape}')
print(f'\nColumnas: {df_fl.columns.tolist()}')

Registros eliminados (fall_les=9): 9
Dimensiones: (71942, 26)

Columnas: ['num_corre', 'año_ocu', 'dia_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymas', 'g_edad_60ymas', 'edad_quinquenales', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', 'fall_les', 'int_o_noint', 'año_carga']


### ( Fallecidos y lesionados normalizado ) : 26 columnas limpias, 71,942 registros

In [10]:
# -- Validar nulos tras limpieza ----------------------------------------------
nulos = df_fl.isnull().sum()
nulos = nulos[nulos > 0]
if len(nulos) == 0:
    print('✓ Sin nulos')
else:
    print(nulos)

dupes = [c for c in df_fl.columns if df_fl.columns.tolist().count(c) > 1]
print(f'Columnas duplicadas: {dupes if dupes else "ninguna"}')

# -- Confirmar fall_les limpio ----------------------------------------------
print(f'\nValores únicos fall_les: {sorted(df_fl["fall_les"].unique())}')

print(f'\nPrimeras 3 filas:')
df_fl.head(3)

✓ Sin nulos
Columnas duplicadas: ninguna

Valores únicos fall_les: [np.int64(1), np.int64(2)]

Primeras 3 filas:


,num_corre,año_ocu,dia_ocu,hora_ocu,g_hora,g_hora_5,mes_ocu,dia_sem_ocu,mupio_ocu,depto_ocu,zona_ocu,sexo_per,edad_per,g_edad_80ymas,g_edad_60ymas,edad_quinquenales,mayor_menor,tipo_veh,marca_veh,color_veh,modelo_veh,g_modelo_veh,tipo_eve,fall_les,int_o_noint,año_carga
0,1,2018,1,16,3,2,1,1,115,1,99,1,28,4,4,6,1,4,32,5,9999,99,2,2,1,2018
1,2,2018,1,12,3,2,1,1,2207,22,99,2,18,2,2,4,1,4,6,1,9999,99,1,2,1,2018
2,3,2018,1,7,2,1,1,1,2102,21,99,1,42,7,7,9,1,1,999,6,9999,99,2,1,2,2018


### ( Fallecidos y lesionados validado ) : fall_les contiene únicamente las clases válidas (1 = Fallecido, 2 = Lesionado).

In [11]:
# -- Guardar fallecidos y lesionados limpio ----------------------------------------------
df_fl.to_csv(os.path.join(OUT_PATH, 'fallecidos_lesionados_clean.csv'), index=False)
print(f'✓ fallecidos_lesionados_clean.csv guardado — {df_fl.shape[0]:,} registros × {df_fl.shape[1]} columnas')

✓ fallecidos_lesionados_clean.csv guardado — 71,942 registros × 26 columnas


In [12]:
# -- Verificar archivos en data/clean ----------------------------------------------
import os

clean_path = '../data/clean'
archivos = os.listdir(clean_path)
print('Archivos en data/clean:')
for f in archivos:
    size = os.path.getsize(os.path.join(clean_path, f)) / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

Archivos en data/clean:
  fallecidos_lesionados_clean.csv  (5.1 MB)
  hechos_clean.csv  (2.8 MB)
  test.parquet  (0.1 MB)
  train.parquet  (0.2 MB)
  vehiculos_involucrados_clean.csv  (5.6 MB)


In [13]:
# -- Cargar y verificación los 3 archivos limpios ----------------------------------------------
df_h  = pd.read_csv('../data/clean/hechos_clean.csv')
df_v  = pd.read_csv('../data/clean/vehiculos_involucrados_clean.csv')
df_fl = pd.read_csv('../data/clean/fallecidos_lesionados_clean.csv')

for nombre, df in [('hechos', df_h), ('vehiculos_involucrados', df_v), ('fallecidos_lesionados', df_fl)]:
    print(f'{nombre}')
    print(f'  Shape     : {df.shape}')
    print(f'  Nulos     : {df.isnull().sum().sum()}')
    print(f'  Duplicados: {df.duplicated().sum()}')
    print()

hechos
  Shape     : (52488, 18)
  Nulos     : 0
  Duplicados: 0

vehiculos_involucrados
  Shape     : (80721, 25)
  Nulos     : 0
  Duplicados: 0

fallecidos_lesionados
  Shape     : (71942, 26)
  Nulos     : 0
  Duplicados: 0

